In [3]:
import pandas as pd
import glob
import os

# The **/*.csv tells Python: "Look in every subfolder for any CSV file"
path = './' 
all_files = glob.glob(os.path.join(path, "**/*.csv"), recursive=True)

print(f"Files found: {len(all_files)}")
# It should now say 'Files found: 12' (or more if you have extra CSVs)

# Then run your merge
df_list = [pd.read_csv(f) for f in all_files]
full_year_df = pd.concat(df_list, ignore_index=True)

Files found: 15


In [4]:
df_list = []
for filename in all_files:
    df = pd.read_csv(filename)
    df_list.append(df)

In [5]:
full_year_df = pd.concat(df_list, ignore_index=True)

In [15]:
full_year_df.to_csv('cyclistic_full_year_2025.csv', index=False)

print(f"Done! Merged {len(all_files)} files into one master CSV.")

Done! Merged 15 files into one master CSV.


In [6]:
full_year_df['ride_length'] = (pd.to_datetime(full_year_df['ended_at']) - 
                               pd.to_datetime(full_year_df['started_at'])).dt.total_seconds() / 60

In [19]:
print(full_year_df['ride_length'].describe())

count    1.662011e+07
mean     1.562433e+01
std      4.793022e+01
min     -5.479480e+01
25%      5.481600e+00
50%      9.494400e+00
75%      1.662040e+01
max      1.574900e+03
Name: ride_length, dtype: float64


In [7]:
full_year_df_cleaned = full_year_df[(full_year_df['ride_length'] > 1) & 
                                    (full_year_df['ride_length'] < 1440)]

In [21]:
print(full_year_df_cleaned['ride_length'].describe())

count    1.631533e+07
mean     1.488011e+01
std      2.870600e+01
min      1.000017e+00
25%      5.666900e+00
50%      9.660167e+00
75%      1.679743e+01
max      1.439976e+03
Name: ride_length, dtype: float64


In [8]:
analysis_results = full_year_df_cleaned.groupby('member_casual')['ride_length'].mean()

print("Average Ride Length (Minutes):")
print(analysis_results)

Average Ride Length (Minutes):
member_casual
casual    19.888086
member    12.133937
Name: ride_length, dtype: float64


In [23]:
full_year_df_cleaned['day_of_week'] = pd.to_datetime(full_year_df_cleaned['started_at']).dt.dayofweek

weekly_trends = full_year_df_cleaned.groupby(['member_casual', 'day_of_week'])['ride_length'].count()
print(weekly_trends)

member_casual  day_of_week
casual         0               668631
               1               653262
               2               637707
               3               745578
               4               919845
               5              1190913
               6               962229
member         0              1510452
               1              1667748
               2              1618401
               3              1700904
               4              1560081
               5              1328526
               6              1151055
Name: ride_length, dtype: int64


In [24]:
weekly_summary = weekly_trends.reset_index()

weekly_summary.to_csv('cyclistic_weekly_summary.csv', index=False)

In [25]:
full_year_df_cleaned.to_csv('cyclistic_cleaned_2025.csv', index=False)

In [9]:
# This groups everything by the dimensions you'll use in your charts
tableau_summary = full_year_df_cleaned.groupby(
    ['member_casual', 'day_of_week', 'rideable_type', 'start_station_name']
).agg(
    total_rides=('ride_id', 'count'),
    avg_duration=('ride_length', 'mean')
).reset_index()

# Export this much smaller file
tableau_summary.to_csv('cyclistic_summary_for_tableau.csv', index=False)

print(f"New file size: {tableau_summary.shape[0]} rows. This will load instantly!")

New file size: 33002 rows. This will load instantly!


In [10]:
final_viz_line_chart = full_year_df_cleaned.groupby(['member_casual', 'day_of_week'])['ride_length'].mean().reset_index()

final_viz_line_chart.to_csv('line_chart_data.csv', index=False)